---
# Section 3: ADK -- Demo 1

**ADK (Agent Development Kit)** -- Google's open-source, model-agnostic framework for AI agents.

| Concept | ADK Implementation |
|---------|-------------------|
| Agent with tools | `Agent(instruction, tools, model)` |
| Multi-agent hierarchy | `sub_agents` parameter |
| Sequential pipeline | `SequentialAgent` |
| Parallel fan-out | `ParallelAgent` |
| Iterative loop | `LoopAgent` |
| LLM-driven routing | Automatic via sub-agent `description` |

### Demo 1 Architecture

```
Router Agent (no tools -- only routes)
  |-- deal_agent       -> lookup_deals, process_claims
  |-- support_agent    -> search_knowledge_base, check_system_status
  |-- escalation_agent -> create_escalation_ticket
```

In [1]:
# Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "google-adk", "google-genai", "python-dotenv", "pywin32", "--ignore-installed", "cffi", "-q"])

0

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

env_path = Path(__vsc_ipynb_file__).parent / ".env" if "__vsc_ipynb_file__" in dir() else Path(".env")
if not env_path.exists():
    env_path = Path(os.getcwd()) / ".env"
load_dotenv(dotenv_path=str(env_path), override=True, encoding="utf-8-sig")
MODEL = "gemini-2.5-flash"
print(f"Ready (loaded from {env_path})" if os.getenv("GOOGLE_API_KEY") else f"WARNING: Set GOOGLE_API_KEY in .env (searched: {env_path})")

### Helper Function

In [ ]:
async def run_agent(agent, message):
    """Run an ADK agent and return the response."""
    service = InMemorySessionService()
    runner = Runner(agent=agent, app_name="demo", session_service=service)
    session = await service.create_session(app_name="demo", user_id="user1")
    content = types.Content(role="user", parts=[types.Part(text=message)])
    async for event in runner.run_async(user_id="user1", session_id=session.id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text
    return "(no response)"

### Step 1: Define Tools

Tools are plain Python functions. The LLM reads docstrings to decide when to call them.

In [ ]:
# Deal tools

def lookup_deals(customer_email: str) -> dict:
    """Look up active deals for a customer by email."""
    deals = {
        "marco@example.com": {"deal_id": "DEAL-2026-101", "customer": "Marco Rivera", "value": "$15,000", "stage": "negotiation", "product": "Enterprise Suite"},
        "sarah@example.com": {"deal_id": "DEAL-2026-102", "customer": "Sarah Chen", "value": "$4,500", "stage": "closed_won", "product": "Professional Plan"},
        "kevin@example.com": {"deal_id": "DEAL-2026-103", "customer": "Kevin Patel", "value": "$8,200", "stage": "proposal_sent", "product": "Team Bundle"},
    }
    return deals.get(customer_email.lower(), {"error": f"No deal found for {customer_email}"})

def process_claims(deal_id: str, reason: str) -> dict:
    """Process a claim or adjustment on a specific deal."""
    return {"claim_id": f"CLM-{deal_id[-3:]}", "status": "under_review", "message": f"Claim for {deal_id} submitted. Review within 2-3 business days."}

# Support tools

def search_knowledge_base(query: str) -> dict:
    """Search the knowledge base for support solutions."""
    articles = {
        "sync": {"title": "Data Sync Failures", "solution": "1. Verify API credentials. 2. Check rate limits. 3. Re-trigger sync from dashboard."},
        "permission": {"title": "Permission Denied Errors", "solution": "1. Confirm user role in admin panel. 2. Clear session tokens. 3. Re-assign workspace access."},
        "export": {"title": "Export Not Working", "solution": "1. Use CSV format for large datasets. 2. Disable ad-blocker. 3. Try a different browser."},
    }
    for keyword, article in articles.items():
        if keyword in query.lower():
            return article
    return {"title": "General Support", "solution": "No specific article found. Please provide more details."}

def check_system_status() -> dict:
    """Check current status of all platform services."""
    return {"overall": "operational", "api_gateway": "healthy", "data_pipeline": "degraded", "last_incident": "2026-03-15"}

# Escalation tool

def create_escalation_ticket(customer_email: str, issue_summary: str, priority: str) -> dict:
    """Create an escalation ticket for issues needing human review."""
    times = {"low": "48 hours", "medium": "24 hours", "high": "4 hours", "critical": "1 hour"}
    return {"ticket_id": "ESC-2026-0187", "priority": priority, "estimated_response": times.get(priority, "24 hours")}

### Step 2: Create Agents

Each agent: name, description (used by router), instruction, and tools.

In [ ]:
deal_agent = Agent(
    name="deal_agent", model=MODEL,
    description="Handles deals: lookups, pricing, claims.",
    instruction="You are a deals specialist. Use lookup_deals and process_claims.",
    tools=[lookup_deals, process_claims],
)

support_agent = Agent(
    name="support_agent", model=MODEL,
    description="Handles support: troubleshooting, outages, knowledge base.",
    instruction="You are a support specialist. Use search_knowledge_base and check_system_status.",
    tools=[search_knowledge_base, check_system_status],
)

escalation_agent = Agent(
    name="escalation_agent", model=MODEL,
    description="Handles complaints, disputes, security concerns.",
    instruction="You are an escalation specialist. Use create_escalation_ticket.",
    tools=[create_escalation_ticket],
)

### Step 3: Create the Router

The router has **no tools**. It reads each sub-agent's `description` to decide who handles each query.

In [ ]:
root_agent = Agent(
    name="customer_support_router", model=MODEL,
    instruction="Route to deal_agent, support_agent, or escalation_agent. Never answer directly.",
    sub_agents=[deal_agent, support_agent, escalation_agent],
)

### Step 4: Test It

In [ ]:
# Test 1: Deal
response = await run_agent(root_agent,
    "I need to check the deal for kevin@example.com. Can we file a claim on it?")
print(response)

In [ ]:
# Test 2: Support
response = await run_agent(root_agent,
    "Our data sync keeps failing and I think there might be an outage. Can you check?")
print(response)

In [ ]:
# Test 3: Escalation
response = await run_agent(root_agent,
    "Unauthorized access on my account! I see suspicious activity. Email: marco@example.com. This is critical!")
print(response)